In [ ]:
import flaml
flaml.__version__

In [ ]:
# %pip list 
%pip show pytorch_forecasting
%pip show flaml
%pip show scikit-learn

If you want to **register model and load model through the mlflow**, ensure you install the package using %pip instead of the code in the next cell.
Example code
```python
# For Spark 3.4
%pip install pytorch-forecasting==1.0.0

# For Spark 3.5
%pip install cython==0.29.37
%pip install pytorch-forecasting==0.10.1 --no-build-isolation
%pip install cython==3.0.10
%pip install torch==2.2.2 torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu
%pip install scikit-learn==1.1.3    # https://github.com/jdb78/pytorch-forecasting/issues/1269
```

In [ ]:
from platform import python_version
from packaging.version import Version
import os
if Version(py_version := python_version()) < Version("3.11"):
    print(os.popen('pip install pytorch-forecasting==1.0.0').read())
else:
    print(os.popen('pip install numpy==1.23.5 cython==0.29.37').read())
    print(os.popen('pip install pytorch-forecasting==0.10.1 --no-build-isolation').read())
    print(os.popen('pip install cython==3.0.10').read())
    print(os.popen('pip install torch==2.2.1 torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu').read())
    print(os.popen('pip install scikit-learn==1.1.3').read())    # https://github.com/jdb78/pytorch-forecasting/issues/1269
f"Current Python Version: {py_version=}"

In [ ]:
%pip list 
%pip show pytorch_forecasting
%pip show scikit-learn

In [ ]:
import mlflow
import os
import logging

mlflow.autolog(disable=True)
logging.getLogger('mlflow.utils.requirements_utils').setLevel(logging.ERROR)

In [ ]:
import flaml
flaml.__version__

# Simple Timeseries Prediction

In [ ]:
import statsmodels.api as sm
data = sm.datasets.co2.load_pandas().data
# data is given in weeks, but the task is to predict monthly, so use monthly averages instead
data = data['co2'].resample('MS').mean()
data = data.bfill().ffill()  # makes sure there are no missing values
data = data.to_frame().reset_index()

In [ ]:
# split the data into a train dataframe and X_test and y_test dataframes, where the number of samples for test is equal to
# the number of periods the user wants to predict
num_samples = data.shape[0]
time_horizon = 12
split_idx = num_samples - time_horizon
train_df = data[:split_idx]  # train_df is a dataframe with two columns: timestamp and label
X_test = data[split_idx:]['index'].to_frame()  # X_test is a dataframe with dates for prediction
y_test = data[split_idx:]['co2']  # y_test is a series of the values corresponding to the dates for prediction

In [ ]:
train_df

import matplotlib.pyplot as plt

plt.plot(train_df['index'], train_df['co2'])
plt.xlabel('Date')
plt.ylabel('CO2 Levels')
plt.show()

In [ ]:
''' import AutoML class from flaml package '''
from flaml import AutoML
automl = AutoML()

settings = {
    "time_budget": 15,  # total running time in seconds
    "metric": 'mape',  # primary metric for validation: 'mape' is generally used for forecast tasks
    "task": 'ts_forecast',  # task type
    "log_file_name": 'CO2_forecast.log',  # flaml log file
    "eval_method": "holdout",  # validation method can be chosen from ['auto', 'holdout', 'cv']
    "seed": 7654321,  # random seed
}

'''The main flaml automl API'''
automl.fit(dataframe=train_df,  # training data
           label='co2',  # label column
           period=time_horizon,  # key word argument 'period' must be included for forecast task)
           **settings)

In [ ]:
''' retrieve best config and best learner'''
print('Best ML leaner:', automl.best_estimator)
print('Best hyperparmeter config:', automl.best_config)
print(f'Best mape on validation data: {automl.best_loss}')
print(f'Training duration of best run: {automl.best_config_train_time}s')

In [ ]:
''' compute predictions of testing dataset '''
flaml_y_pred = automl.predict(X_test)
print(f"Predicted labels\n{flaml_y_pred}")
print(f"True labels\n{y_test}")

''' compute different metric values on testing dataset'''
from flaml.ml import sklearn_metric_loss_score
print('mape', '=', sklearn_metric_loss_score('mape', y_true=y_test, y_predict=flaml_y_pred))

In [ ]:
from flaml.automl.data import get_output_from_log
time_history, best_valid_loss_history, valid_loss_history, config_history, train_loss_history = \
    get_output_from_log(filename=settings['log_file_name'], time_budget=180)

for config in config_history:
    print(config)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plt.title('Learning Curve')
plt.xlabel('Wall Clock Time (s)')
plt.ylabel('Validation Accuracy')
plt.scatter(time_history, 1 - np.array(valid_loss_history))
plt.step(time_history, 1 - np.array(best_valid_loss_history), where='post')
plt.show()

In [ ]:
import matplotlib.pyplot as plt
plt.plot(X_test, y_test, label='Actual level')
plt.plot(X_test, flaml_y_pred, label='FLAML forecast')
plt.xlabel('Date')
plt.ylabel('CO2 Levels')
plt.legend()

# Panel Dataset Timeseries prediction

- Q1: **Panel Time Series**: They want to train a model using many rows for the same year & month. However, they were only able to get the timeseries forecast working using a specific combination of Unit, Department, Accounts. Is it possible to train a time series with more than one value for a given time period?
   - Here is a panel dataset prediction by diffenrent department(agency), accounts(sku) with year & month(the time is set to be the first day of the month)
- Q2: **Support for Text Features**: They were wondering if AutoML for forecasting would work with a dataframe containing text feature columns. They kept hitting issues with this but were successful when they eliminated the text columns.
   - Encoder the text accordingly. The text data need to be transformed to numerical 
   data for traditional machine learning.
- Q3: **Sequential Dates**: They wanted to train using year and month, however we were only able to get this to work by creating a column with sequential dates. Is it possible to train a time series forecast with year and month instead of sequential dates? If not, could they have sequenced the data with whole numbers instead of dates?  Like a DENSE_RANK??
   - The year and month are sequential data, I assert simply convert the year and 
   month to datetime64 will work
- Q4.	**Best Practices**: They had a few questions on the best practices on determining how to set their time budget and the implications with a lower time budget.
- Q5.	**Log File**: Where is the flaml log file stored?
- Q6.	**Models Explored**: How do we know which models are evaluated?
   - The evaluated models shown in the log. All the collected models is in INFO - List of ML learners in AutoML Run: [...]
- Q7: **[Issue] – missing internal packages**: The error about synapse-mlflow and synapseml-internal is because the packages
   is not public, maybe mlflow can filter the package warning? But set mlflow.autolog(exclusive=False)
       simply works.
   - Q7-2: In addition, there are some messages that certain libraries like pytorch_lightning, prophet, and orbit couldn’t be imported. I believe this is because we decided that users need to install these dependencies – do you remember why we didn’t include them by default? I recall one had a GPU dependency? Perhaps, we can give a better inline message with the reason they couldn’t be imported so users know if they want to install it themselves. Please let me know the cases for these 3 libraries and I can come up with an error message.
   - prophet and pytorch_lightning is setup in the extra option flaml[forecast] the
   notebook work for me by using pip install flaml[spark,automl,forecast]
- Q8: **[Issue] – missing MLflow widget**
- Q9.	**Metrics Selection**
   - MAPE is the default and relativly enough. All supported metrics can be seen in "flaml/automl/ml.py:43-91"

- **When using a csv file we had to cast columns from object to specific types. Do you know a better way to do this?**
    - The "withColumn" is enough. Maybe you can transfer to pandas dataframe and use "map" or "df.astype({"col_name": type_})"
- **Also, we could only get the to_date to work by setting to M/d/y.  Why doesn't M/d/yyyy work?**
    - You can refer to "https://docs.oracle.com/javase/tutorial/i18n/format/simpleDateFormat.html" and "https://docs.oracle.com/javase/8/docs/api/java/text/SimpleDateFormat.html" for more information about datetime format
- **Are the settings considered "hyperparameters" e.g. MAPE metric?  Or is the AutoML technically choosing the hyperparameters for the model selected?**
    - MAPE is the default and relativly enough. All the sklearn and huggingface supported metrics can be used. 
    You can try different metrics to get better performance.
- **How do we know which models are evaluated?  Looking at the log below?  What about Holt-Winters exponential smoothing?**
    - The evaluated models shown in the log. All the collected models is in INFO - List of ML learners in AutoML Run: [...]
- **Do we have an environment to install synapseml-mlflow and synapseeml-internal or do we need to pip install them?**
    - The warning comes from that the packages are not open-sourced, no need to install. set mlflow.autolog(**, silent=True) 
    if you don't want to see the warning in normal run.
- **A ROC curve is a plot of the false alarm rate (also known as probability of false detection or POFD) on the x-axis, versus the hit-rate (also known as probability of detection-yes or PODy) on the y-axis.
  Is this only applicable to classification models?**
    - It is applicable to binary classification task.

In [ ]:
"success"

In [ ]:
def get_stalliion_data():
    from pytorch_forecasting.data.examples import get_stallion_data

    data = get_stallion_data()
    # data.rename(columns={"sku": "accounts", "agency": "department"}, inplace=True)
    data.pop('industry_volume')
    data.pop('soda_volume')
    data.pop('avg_max_temp')
    data.pop('price_regular')
    data.pop('discount_in_percent')
    # we want to encode special days as one variable and thus need to first reverse one-hot encoding
    special_days = [
        "easter_day",
        "good_friday",
        "new_year",
        "christmas",
        "labor_day",
        "independence_day",
        "revolution_day_memorial",
        "regional_games",
        "beer_capital",
        "music_fest",
    ]
    data[special_days] = (
        data[special_days]
        .apply(lambda x: x.map({0: "-", 1: x.name}))
        .astype("category")
    )
    return data, special_days

In [ ]:
import numpy as np
data, special_days = get_stalliion_data()
time_horizon = 6  # predict six months
# make time steps first column
data["time_idx"] = data["date"].dt.year * 12 + data["date"].dt.month
data["time_idx"] -= data["time_idx"].min()
training_cutoff = data["time_idx"].max() - time_horizon
ts_col = data.pop("date")
data.insert(0, "date", ts_col.apply(lambda x: np.datetime64(x, "ns")))
# FLAML assumes input is not sorted, but we sort here for comparison purposes with y_test
data = data.sort_values(["agency", "sku", "date"])
X_train = data[lambda x: x.time_idx <= training_cutoff]
X_test = data[lambda x: x.time_idx > training_cutoff]
y_train = X_train.pop("volume")
y_test = X_test.pop("volume")

In [ ]:
X_train.sort_values(["date", "agency", "sku"])

In [ ]:
X_train.dtypes

In [ ]:

from flaml import AutoML
automl = AutoML()
settings = {
    "time_budget": 100,  # total running time in seconds
    "metric": "mape",  # primary metric
    "task": "ts_forecast_panel",  # task type
    "log_file_name": "stallion_forecast.log",  # flaml log file
    "eval_method": "holdout",
}
fit_kwargs_by_estimator = {        # adding time varying known and unkown variables for your dataset.
    "tft": {
        "max_encoder_length": 12,
        "static_categoricals": ["agency", "sku"], # Here is panel things
        "time_varying_known_reals": ["time_idx" ], # time index is needed
        "time_varying_unknown_reals": ["volume"], # always need a target column
        "max_epochs": 1,
    }
}
# # """The main flaml automl API"""
# automl.fit(
#     X_train=X_train,
#     y_train=y_train,
#     **settings,
#     period=time_horizon,
#     group_ids=["agency", "sku"],
#     fit_kwargs_by_estimator=fit_kwargs_by_estimator,
# )

In [ ]:
# %pip install scikit-learn==1.0.2

In [ ]:
# Set given experiment as the active experiment. If an experiment with this name does not exist, a new experiment with this name is created.
# mlflow.set_experiment("jb_time_series_forecast_experiments_panel")
#mlflow.autolog(exclusive=False)
import mlflow
import os
import logging

mlflow.autolog(disable=True)
logging.getLogger('mlflow.utils.requirements_utils').setLevel(logging.ERROR)

with mlflow.start_run(nested=True):
    automl.fit(
        X_train=X_train,
        y_train=y_train,
        **settings,
        period=time_horizon,
        group_ids=["agency", "sku"],
        fit_kwargs_by_estimator=fit_kwargs_by_estimator,
    )

In [ ]:
import sys
sys.version, sys.executable

In [ ]:
""" compute predictions of testing dataset """
y_pred = automl.predict(X_test)
print(y_test)
print(y_pred)

In [ ]:
""" compute different metric values on testing dataset"""
from flaml.ml import sklearn_metric_loss_score
print("mape", "=", sklearn_metric_loss_score("mape", y_pred, y_test))

def smape(y_pred, y_test):
    import numpy as np

    y_test, y_pred = np.array(y_test), np.array(y_pred)
    return round(
        np.mean(
            np.abs(y_pred - y_test) /
            ((np.abs(y_pred) + np.abs(y_test)) / 2)
        ) * 100, 2
    )

print("smape", "=", smape(y_pred, y_test))

In [ ]:
sc.getConf().get("spark.synapse.vhd.id")